# Replication script for the NFM special issue experiments

Run all cells with the notebook directory as the working directory. Required packages: numpy, pandas,
matplotlib (3.10 or newer), scipy and IPython.

- **RQ1/RQ2:** coverage vs mutation on EvoAvalla, Random and ATGT, with mixed configuratio for the first two (use noshuffl entries for the models in `model_list_choose.txt`).
- **RQ3:** ATGT vs random4atgt, with equal scenario and step counts.
- **RQ4:** pick vs noshuffle on the choose models, separately for EvoAvalla and Random.
- **RQ5:** aggregate original vs optimized suites across ATGT, EvoAvalla and Random, using the mixed configuration for EvoAvalla and Random.

## Data preparation

RQ analyses retain only `OK` executions, excluding `NO_VALID_SCENARIOS`, `MUTATION_ERR`, and all other failed statuses.

Aggregate coverage is the mean of the available coverage criteria.
Aggregate mutation scorse is number of killed mutants / total number of mutants, over all mutation operators. 

In [58]:
# Import libraries
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from IPython.display import display
from scipy.stats import linregress, spearmanr, wilcoxon

alpha = 0.05 # For Wilcoxon test significance

base = Path(".")
image_dir = base / "images"
image_dir.mkdir(exist_ok=True)

cov_cols = ["macro_coverage", "update_rule_coverage", "forall_rule_coverage",
            "branch_coverage", "rule_coverage"]
size_cols = ["n_scenarios", "n_step", "n_set", "n_check"]

mutation_names = {
    "ruleremover": "rule_remover", "chooserulemutator": "choose_remover",
    "condremover": "condition_remover", "condnegator": "condition_negator",
    "forallmutator": "forall_mutator", "partoseqmutator": "par_to_seq",
    "seqtoparmutator": "seq_to_par", "casemutator": "case_mutator",
}

def model_key(value):
    return str(value).strip().replace("\\", "/").split("asm_examples/", 1)[-1]

def read_models(filename):
    lines = (base / filename).read_text(encoding="utf-8-sig").splitlines()
    models = [model_key(line.split("//", 1)[0]) for line in lines
              if line.split("//", 1)[0].strip()]
    if len(models) != len(set(models)):
        raise ValueError(f"Duplicate paths in {filename}")
    return models

def show_table(frame, title):
    print("\n" + title.replace("_", " "))
    if frame.empty:
        print("No entries.")
    else:
        shown = frame.copy()
        # Format coverage and mutation values without changing the underlying data
        quality_fields = ["baseline_mean", "candidate_mean", "mean_delta", "mean_delta_pp"]
        if "metric" in shown.columns:
            quality = shown["metric"].isin(quality_cols)
            for column in quality_fields:
                if column in shown.columns and quality.any():
                    shown[column] = shown[column].astype(object)
                    shown.loc[quality, column] = shown.loc[quality, column].map(
                        lambda value: f"{value:.2f}" if pd.notna(value) else "NaN")
        if "mean_mutation_delta" in shown.columns:
            shown["mean_mutation_delta"] = shown["mean_mutation_delta"].map(
                lambda value: f"{value:.2f}" if pd.notna(value) else "NaN")
        print(shown.to_string(index=False, float_format=lambda value: f"{value:.6g}"))
    print()

model_list = read_models("model_list.txt")
model_list_choose = read_models("model_list_choose.txt")
choose_models = set(model_list_choose)

raw = pd.read_csv(base / "data.csv")
required = ["asm_path", "approach", "status", *cov_cols, *size_cols]
if not set(required) <= set(raw.columns):
    raise ValueError(f"Missing columns: {set(required) - set(raw.columns)}")
if raw[["asm_path", "approach", "status"]].isna().any().any():
    raise ValueError("Missing model path, approach or execution status")
raw["asm_path"] = raw["asm_path"].map(model_key)
raw["approach"] = raw["approach"].str.strip()
raw["status"] = raw["status"].str.strip()

# Discover all mutation operators
count_ops = {c[:-6] for c in raw if c.endswith("_count")}
killed_ops = {c[:-7] for c in raw if c.endswith("_killed")}
if count_ops != killed_ops or not count_ops:
    raise ValueError(f"Unmatched mutation columns: {count_ops ^ killed_ops}")
operators = [c[:-6] for c in raw if c.endswith("_count")]
mut_cols = [mutation_names.get(op, op) for op in operators]
mut_count_cols = [op + "_count" for op in operators]
mut_killed_cols = [op + "_killed" for op in operators]
numeric_cols = cov_cols + size_cols + mut_count_cols + mut_killed_cols
for col in numeric_cols:
    raw[col] = pd.to_numeric(raw[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

scores = {}
if len(mut_cols) != len(set(mut_cols)):
    raise ValueError("Mutation operator names are not unique after renaming")
for op, score in zip(operators, mut_cols):
    count, killed = raw[op + "_count"], raw[op + "_killed"]
    invalid = ((count < 0) | (killed < 0) | (killed > count))
    if invalid.any():
        raise ValueError(f"Invalid mutation counts for {op}: rows {raw.index[invalid].tolist()}")
    scores[score] = killed / count.where(count > 0)
raw = pd.concat([raw, pd.DataFrame(scores, index=raw.index)], axis=1)

if ((raw[cov_cols] < 0) | (raw[cov_cols] > 1)).any().any():
    raise ValueError("Coverage must be in [0, 1]")
num = raw[mut_killed_cols].sum(axis=1, skipna=True)
den = raw[mut_count_cols].sum(axis=1, skipna=True)
raw["agg_mut_general"] = (num / den).mask(den == 0)
raw["agg_cov_mean"] = raw[cov_cols].mean(axis=1)
mut_metric = "agg_mut_general"
quality_cols = cov_cols + ["agg_cov_mean", mut_metric]

df = raw[raw["asm_path"].isin(model_list)].copy()
print(f"Models: {len(model_list)}, of which {len(choose_models)} with choose.\nNumber of mutation operators: {len(operators)}")

Models: 295, of which 99 with choose.
Number of mutation operators: 27


In [59]:
# Get a copy of the DataFrame filtered by approach, mode (pick, noshuffle or mixed), and optimization.
def select_approach(tool, mode="pick", optimized=False):
    prefix = "opt_" if optimized else ""
    if tool in ("atgt", "random4atgt"):
        selected = df[df["approach"] == prefix + tool]
    elif mode == "mixed":
        choose = df["asm_path"].isin(choose_models)
        selected = df[((df["approach"] == prefix + tool + "_noshuffle") & choose)
                      | ((df["approach"] == prefix + tool + "_pick") & ~choose)]
    elif mode in ("pick", "noshuffle"):
        selected = df[df["approach"] == prefix + tool + "_" + mode]
        if mode == "noshuffle":
            selected = selected[selected["asm_path"].isin(choose_models)]
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return selected.copy()

# expected keep track of which entries are expected for each approach.
expected = {"atgt": model_list, "atgtold": model_list, "opt_atgt": model_list}
atgt_ok_models = select_approach("atgt").query("status == 'OK'")["asm_path"].tolist()
expected["random4atgt"] = model_list
for tool in ("evoavalla", "random"):
    for prefix in ("", "opt_"):
        expected[prefix + tool + "_pick"] = model_list
        expected[prefix + tool + "_noshuffle"] = model_list_choose



## Initial consistency checks

Each available approach must have one row per expected model: 295 for pick/ATGT (both new and old versions),
99 for noshuffle, and 295 for random4atgt. Same for optimized entries.

In [60]:
# Check one entry per expected model, before filtering by status
summary, entry_issues = [], []
for approach, scope in expected.items():
    runs = raw[raw["approach"] == approach]
    counts = runs.groupby("asm_path").size()
    missing = set(scope) - set(counts.index) if not runs.empty else set()
    extra = set(counts.index) - set(scope)
    duplicates = set(counts[counts > 1].index)
    summary.append([approach, len(scope), len(runs), len(missing), len(duplicates), len(extra),
                    "PENDING" if runs.empty else "FAIL" if missing or extra or duplicates else "OK"])
    for problem, models in [("missing", missing), ("duplicate", duplicates), ("outside list", extra)]:
        entry_issues.extend((approach, model, problem) for model in sorted(models))

show_table(pd.DataFrame(summary, columns=["approach", "expected", "rows", "missing", "duplicates", "extra", "check"]),
           "Entry checks")
entry_issues = pd.DataFrame(entry_issues, columns=["approach", "asm_path", "problem"])
if not entry_issues.empty:
    show_table(entry_issues.head(5), "Entry errors - first 5")
    raise ValueError(f"Initial validation failed: {len(entry_issues)} entry issues")

# Check that, when ATGT is OK, random4atgt has at least one scenario.
random_runs = select_approach("random4atgt")
invalid_scenarios = random_runs.loc[
    ~random_runs["asm_path"].isin(atgt_ok_models)
    & random_runs["n_scenarios"].notna()
    & random_runs["n_scenarios"].ne(0),
    ["asm_path", "status", "n_scenarios"],
]
if not invalid_scenarios.empty:
    show_table(invalid_scenarios.head(5), "random4atgt scenarios without ATGT OK - first 5")
    raise ValueError(f"random4atgt has nonzero scenarios without ATGT OK for {len(invalid_scenarios)} models")

print("Initial entry checks passed for all available approaches.")



Entry checks
               approach  expected  rows  missing  duplicates  extra check
                   atgt       295   295        0           0      0    OK
                atgtold       295   295        0           0      0    OK
               opt_atgt       295   295        0           0      0    OK
            random4atgt       295   295        0           0      0    OK
         evoavalla_pick       295   295        0           0      0    OK
    evoavalla_noshuffle        99    99        0           0      0    OK
     opt_evoavalla_pick       295   295        0           0      0    OK
opt_evoavalla_noshuffle        99    99        0           0      0    OK
            random_pick       295   295        0           0      0    OK
       random_noshuffle        99    99        0           0      0    OK
        opt_random_pick       295   295        0           0      0    OK
   opt_random_noshuffle        99    99        0           0      0    OK

Initial entry checks pa

In [61]:
'''
# new vs old ATGT comparison
tmp_atgt_versions = {
    name: df[df["approach"].eq(name)].set_index("asm_path").reindex(model_list)
    for name in ("atgt", "atgtold")
}
tmp_atgt_support = pd.DataFrame([
    {
        "approach": name, "expected": len(model_list),
        "present": runs["status"].notna().sum(),
        "OK": runs["status"].eq("OK").sum(),
        "failed": (runs["status"].notna() & runs["status"].ne("OK")).sum(),
        "missing": runs["status"].isna().sum(),
    }
    for name, runs in tmp_atgt_versions.items()
])
show_table(tmp_atgt_support, "new vs old ATGT: successful specifications")
tmp_atgt_new, tmp_atgt_old = tmp_atgt_versions["atgt"], tmp_atgt_versions["atgtold"]
tmp_atgt_both_ok = tmp_atgt_new["status"].eq("OK") & tmp_atgt_old["status"].eq("OK")
print(f"Specifications OK in both versions: {tmp_atgt_both_ok.sum()}")
if not tmp_atgt_both_ok.any():
    print("Comparison pending: no specifications are currently OK in both versions.")
else:
    tmp_atgt_rows = []
    for metric in quality_cols:
        # Exclude missing values jointly so both means use exactly the same models.
        values = pd.concat([
            tmp_atgt_new.loc[tmp_atgt_both_ok, metric].rename("new"),
            tmp_atgt_old.loc[tmp_atgt_both_ok, metric].rename("old"),
        ], axis=1).dropna()
        tmp_atgt_rows.append({
            "metric": metric, "common_OK": int(tmp_atgt_both_ok.sum()),
            "complete_pairs": len(values),
            "missing_metric_pairs": int(tmp_atgt_both_ok.sum()) - len(values),
            "new_mean_pct": 100 * values["new"].mean(),
            "old_mean_pct": 100 * values["old"].mean(),
            "new_minus_old_pp": 100 * (values["new"] - values["old"]).mean(),
        })
    tmp_atgt_comparison = pd.DataFrame(tmp_atgt_rows)
    print("mean coverage and mutation score on common OK specifications")
    print("Means are percentages; positive deltas favor the new ATGT version.")
    print(tmp_atgt_comparison.to_string(index=False, float_format=lambda value: f"{value:.2f}"))
'''

'\n# new vs old ATGT comparison\ntmp_atgt_versions = {\n    name: df[df["approach"].eq(name)].set_index("asm_path").reindex(model_list)\n    for name in ("atgt", "atgtold")\n}\ntmp_atgt_support = pd.DataFrame([\n    {\n        "approach": name, "expected": len(model_list),\n        "present": runs["status"].notna().sum(),\n        "OK": runs["status"].eq("OK").sum(),\n        "failed": (runs["status"].notna() & runs["status"].ne("OK")).sum(),\n        "missing": runs["status"].isna().sum(),\n    }\n    for name, runs in tmp_atgt_versions.items()\n])\nshow_table(tmp_atgt_support, "new vs old ATGT: successful specifications")\ntmp_atgt_new, tmp_atgt_old = tmp_atgt_versions["atgt"], tmp_atgt_versions["atgtold"]\ntmp_atgt_both_ok = tmp_atgt_new["status"].eq("OK") & tmp_atgt_old["status"].eq("OK")\nprint(f"Specifications OK in both versions: {tmp_atgt_both_ok.sum()}")\nif not tmp_atgt_both_ok.any():\n    print("Comparison pending: no specifications are currently OK in both versions.")\nelse

## RQ1 and RQ2: Coverage criteria and fault detection capability correlation

We use the **mixed** configuration: noshuffle for the 99 models containing choose and pick for the remaining models (which should be equivalente to noshuffle). ATGT is unchanged.
Each successful model/approach contributes one point.
Results combine the OK executions from ATGT, EvoAvalla and Random.

For each coverage criterion (and mean coverage), report linear regression and Spearman correlation on all valid points against the aggregated mutation score.

In [62]:
def correlation_stats(executions, cov):
    pairs = executions[[cov, mut_metric]].dropna()
    x, y = pairs[cov], pairs[mut_metric]
    result = {"n": len(pairs), "coverage_100": x.eq(1).sum(),
              "slope": np.nan, "intercept": np.nan, "r_squared": np.nan,
              "regression_p": np.nan, "spearman_rho": np.nan, "spearman_p": np.nan}
    if len(pairs) >= 2 and x.nunique() > 1 and y.nunique() > 1:
        res = linregress(x, y)
        rho, rho_p = spearmanr(x, y)
        result.update(slope=res.slope, intercept=res.intercept, r_squared=res.rvalue**2,
                      regression_p=res.pvalue, spearman_rho=rho, spearman_p=rho_p)
    return result

rq12_executions = {}
rq12_results = {}
for mode in ("mixed",):
    print("--------------------")
    print(f"RQ1/RQ2 — {mode}")
    print("--------------------")
    approach_executions = [select_approach(tool, mode) for tool in ("atgt", "evoavalla", "random")]
    combined_executions = pd.concat([executions[executions["status"] == "OK"] for executions in approach_executions], ignore_index=True)
    rq12_executions[mode] = combined_executions
    rows = [{"configuration": mode, "coverage": cov, **correlation_stats(combined_executions, cov)}
            for cov in cov_cols + ["agg_cov_mean"]]
    rq12_results[mode] = pd.DataFrame(rows)
    for _, row in rq12_results[mode].iterrows():
        print(f"\n{row['coverage']}")
        print(f"n={row['n']}, 100% coverage={row['coverage_100']}, "
              f"slope={row['slope']:.6g}, R²={row['r_squared']:.6f}, p={row['regression_p']:.3g}, "
              f"Spearman ρ={row['spearman_rho']:.3f}, pρ={row['spearman_p']:.3g}")

    # Plot combined_executions points and the fitted regression, following the original notebook
    for i, cov in enumerate(cov_cols + ["agg_cov_mean"]):
        pairs = combined_executions[[cov, mut_metric]].dropna()
        stats = correlation_stats(combined_executions, cov)
        color = plt.cm.tab10(i)
        fig, ax = plt.subplots(figsize=(2.5, 4))
        ax.scatter(pairs[cov], pairs[mut_metric], s=15, alpha=0.65, edgecolors="none", color=color)
        if pd.notna(stats["slope"]):
            x_line = np.linspace(pairs[cov].min(), pairs[cov].max(), 200)
            ax.plot(x_line, stats["intercept"] + stats["slope"] * x_line, color=color, linewidth=1.5)
        ax.set(xlim=(0, 1), ylim=(0, 1))
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.grid(True, linewidth=0.5, alpha=0.4)
        fig.tight_layout()
        fig.savefig(image_dir / f"scatterplot_{cov}.pdf", bbox_inches="tight")
        plt.close(fig)

--------------------
RQ1/RQ2 — mixed
--------------------

macro_coverage
n=346, 100% coverage=272, slope=0.790966, R²=0.366920, p=5.01e-36, Spearman ρ=0.456, pρ=3.77e-19

update_rule_coverage
n=345, 100% coverage=176, slope=0.666443, R²=0.458837, p=1.17e-47, Spearman ρ=0.560, pρ=6.93e-30

forall_rule_coverage
n=68, 100% coverage=1, slope=0.853321, R²=0.150241, p=0.00109, Spearman ρ=0.341, pρ=0.00449

branch_coverage
n=338, 100% coverage=98, slope=0.809188, R²=0.481448, p=7.6e-50, Spearman ρ=0.553, pρ=1.78e-28

rule_coverage
n=346, 100% coverage=215, slope=0.796452, R²=0.487351, p=7.54e-52, Spearman ρ=0.575, pρ=7.32e-32

agg_cov_mean
n=346, 100% coverage=77, slope=0.869987, R²=0.494434, p=6.84e-53, Spearman ρ=0.595, pρ=1.66e-34


## Method utilities for RQ3–RQ5

Prepare the methods for comparing paired results.
The comparisons summarize changes and use statistical tests to assess their significance.

In [63]:
# Match the two approaches by model, retaining missing and failed executions
def pair_runs(baseline, candidate, scope):
    left = baseline.set_index("asm_path").reindex(scope)
    right = candidate.set_index("asm_path").reindex(scope)
    return left.add_suffix("_baseline").join(right.add_suffix("_candidate")).rename_axis("asm_path")

# Report execution statuses and select models that are OK in both approaches
def report_support(pairs, name, baseline_name, candidate_name):
    left = pairs["status_baseline"].fillna("MISSING")
    right = pairs["status_candidate"].fillna("MISSING")
    both_ok = left.eq("OK") & right.eq("OK")
    baseline_ok = left.eq("OK").sum()
    summary = pd.DataFrame([{
        "comparison": name, "scope": len(pairs),
        f"{baseline_name}_present": pairs["status_baseline"].notna().sum(),
        f"{candidate_name}_present": pairs["status_candidate"].notna().sum(),
        f"{baseline_name}_OK": baseline_ok, f"{candidate_name}_OK": right.eq("OK").sum(),
        "both_OK": both_ok.sum(),
        f"{baseline_name}_OK_retained_pct": 100 * both_ok.sum() / baseline_ok if baseline_ok else np.nan,
        f"{baseline_name}_OK_{candidate_name}_failed": (left.eq("OK") & right.ne("OK") & right.ne("MISSING")).sum(),
        f"{baseline_name}_OK_{candidate_name}_missing": (left.eq("OK") & right.eq("MISSING")).sum(),
        f"{candidate_name}_OK_{baseline_name}_not_OK": (left.ne("OK") & right.eq("OK")).sum(),
    }])
    show_table(summary, name + "_support")
    transitions = pd.DataFrame({f"{baseline_name}_status": left, f"{candidate_name}_status": right})
    show_table(transitions.value_counts().rename("models").reset_index(), name + "_transitions")
    return pairs.loc[both_ok].copy()

# Test paired differences with Wilcoxon, handling empty and all-zero samples
def signed_rank(diff, alternative="two-sided"):
    diff = np.asarray(diff, dtype=float)
    if not len(diff):
        return {"wilcoxon_W": np.nan, "wilcoxon_p": np.nan, "test_note": "no complete pairs"}
    if not np.any(diff):
        return {"wilcoxon_W": 0.0, "wilcoxon_p": 1.0, "test_note": "all differences are zero"}
    res = wilcoxon(diff, alternative=alternative, zero_method="wilcox", method="auto")
    return {"wilcoxon_W": res.statistic, "wilcoxon_p": res.pvalue, "test_note": "-"}

# Summarize paired metric changes and test their significance
def compare_metrics(pairs, metrics, name):
    rows = []
    for metric in metrics:
        cols = [metric + "_baseline", metric + "_candidate"]
        values = pairs[cols].dropna()
        left, right = values[cols[0]], values[cols[1]]
        diff = (right - left)
        test = signed_rank(diff)
        score = metric in quality_cols
        rows.append({
            "metric": metric, "n_common_OK": len(pairs), "n_pairs": len(values),
            "missing_metric_pairs": len(pairs) - len(values),
            "baseline_mean": left.mean(), "candidate_mean": right.mean(),
            "mean_delta": diff.mean(),
            "mean_delta_pp": 100 * diff.mean() if score else np.nan,
            "increased": (diff > 0).sum(), "unchanged": (diff == 0).sum(),
            "decreased": (diff < 0).sum(), "n_nonzero": (diff != 0).sum(), **test,
            "significant": bool(test["wilcoxon_p"] < alpha) if pd.notna(test["wilcoxon_p"]) else pd.NA,
        })
    result = pd.DataFrame(rows)
    descriptive = ["metric", "n_pairs", "missing_metric_pairs", "baseline_mean", "candidate_mean",
                   "mean_delta", "mean_delta_pp"]
    show_table(result[descriptive], name + "_descriptive")
    tests = ["metric", "n_pairs", "increased", "unchanged", "decreased", "n_nonzero",
             "wilcoxon_W", "wilcoxon_p", "significant", "test_note"]
    show_table(result[tests], name + "_tests")

    return result

## RQ3: Coverage value and fault detection capability (with equal length)

For each coverage criterion, select models where
ATGT coverage is greater than random4atgt coverage, then test **ATGT − random4atgt mutation
score** using a one-sided Wilcoxon signed-rank test.

The ATGT-supported subset is derived from its `OK` rows. Both runs must be `OK`, with
nonmissing, equal `n_scenarios` and `n_step`. Mismatches are reported and excluded.

In [64]:
atgt = select_approach("atgt")
random4atgt = select_approach("random4atgt")
rq3_pairs = pair_runs(atgt, random4atgt, model_list)
rq3_common = report_support(rq3_pairs, "rq3_atgt_random4atgt", "atgt", "random4atgt")
equal_size = pd.Series(True, index=rq3_common.index)
for metric in ("n_scenarios", "n_step"):
    equal_size &= (rq3_common[metric + "_baseline"].notna()
                   & rq3_common[metric + "_candidate"].notna()
                   & rq3_common[metric + "_baseline"].eq(rq3_common[metric + "_candidate"]))
show_table(rq3_common.loc[~equal_size, ["n_scenarios_baseline", "n_scenarios_candidate", "n_step_baseline", "n_step_candidate"]].reset_index(), "Unequal or missing suite sizes")
rq3_equal = rq3_common.loc[equal_size]
print(f"RQ3: {len(rq3_common)} common OK models; {len(rq3_equal)} with equal scenario and step counts")
rq3_rows = []
for cov in cov_cols + ["agg_cov_mean"]:
    valid_cov = rq3_equal[[cov + "_baseline", cov + "_candidate"]].notna().all(axis=1)
    greater = valid_cov & (rq3_equal[cov + "_baseline"] > rq3_equal[cov + "_candidate"])
    values = rq3_equal.loc[greater, [mut_metric + "_baseline", mut_metric + "_candidate"]].dropna()
    left, right = values[mut_metric + "_baseline"], values[mut_metric + "_candidate"]
    diff = (left - right)
    row = {"coverage": cov, "n_equal_size": len(rq3_equal), "n_coverage_pairs": valid_cov.sum(),
           "n_atgt_coverage_greater": greater.sum(), "n_mutation_pairs": len(values),
           "mean_mutation_delta": diff.mean(), **signed_rank(diff, alternative="greater")}
    rq3_rows.append(row)
rq3_results = pd.DataFrame(rq3_rows)
show_table(rq3_results, "rq3_tests")


rq3 atgt random4atgt support
          comparison  scope  atgt_present  random4atgt_present  atgt_OK  random4atgt_OK  both_OK  atgt_OK_retained_pct  atgt_OK_random4atgt_failed  atgt_OK_random4atgt_missing  random4atgt_OK_atgt_not_OK
rq3_atgt_random4atgt    295           295                  295       60              60       60                   100                           0                            0                           0


rq3 atgt random4atgt transitions
       atgt_status random4atgt_status  models
NO_VALID_SCENARIOS NO_VALID_SCENARIOS     235
                OK                 OK      60


Unequal or missing suite sizes
No entries.

RQ3: 60 common OK models; 60 with equal scenario and step counts

rq3 tests
            coverage  n_equal_size  n_coverage_pairs  n_atgt_coverage_greater  n_mutation_pairs mean_mutation_delta  wilcoxon_W  wilcoxon_p test_note
      macro_coverage            60                60                       26                26                0.17  

## RQ4: Pick vs noshuffle on choose specifications

First count how many choose models are supported by pick and noshuffle, including the
intersection and status transitions. Then compare mean coverage and general mutation score on the subset
that is **OK in both modes**, separately for EvoAvalla and Random.

Baseline = **noshuffle**, candidate = **pick**. A positive delta indicates higher quality
with pick; a negative delta indicates higher quality with noshuffle. Each metric has a
two-sided Wilcoxon test on its complete pairs. Mutation uses all operators.

In [65]:
rq4_results = {}
for tool in ("evoavalla", "random"):
    print(f"{tool}: baseline=noshuffle, candidate=pick")
    pairs = pair_runs(select_approach(tool, "noshuffle"), select_approach(tool, "pick"), model_list_choose)
    common = report_support(pairs, f"rq4_{tool}", "noshuffle", "pick")
    rq4_results[tool] = compare_metrics(common, ["agg_cov_mean", "agg_mut_general"], f"rq4_{tool}")

evoavalla: baseline=noshuffle, candidate=pick

rq4 evoavalla support
   comparison  scope  noshuffle_present  pick_present  noshuffle_OK  pick_OK  both_OK  noshuffle_OK_retained_pct  noshuffle_OK_pick_failed  noshuffle_OK_pick_missing  pick_OK_noshuffle_not_OK
rq4_evoavalla     99                 99            99            28       18       18                    64.2857                        10                          0                         0


rq4 evoavalla transitions
  noshuffle_status        pick_status  models
NO_VALID_SCENARIOS NO_VALID_SCENARIOS      68
                OK                 OK      18
                OK NO_VALID_SCENARIOS      10
      MUTATION_ERR NO_VALID_SCENARIOS       3


rq4 evoavalla descriptive
         metric  n_pairs  missing_metric_pairs baseline_mean candidate_mean mean_delta mean_delta_pp
   agg_cov_mean       18                     0          0.83           0.74      -0.09         -9.00
agg_mut_general       18                     0          0.4

## RQ5: Original vs optimized suites

Pool the paired original and optimized suites from ATGT, EvoAvalla and Random into one aggregate analysis. EvoAvalla and Random use noshuffle on the choose models and pick on the rest.

Among original `OK` suites, optimization failed if the optimized status is `NO_VALID_SCENARIOS`, or if both statuses are `OK` but the number of tests (`n_scenarios`) has decreased.
An `OK` optimized suite with at least as many tests is considered successful.
Only successful pairs are used to compare set and check counts, mean coverage and general mutation score, with a two-sided Wilcoxon test for each metric.
Deltas are **optimized − original**.

In [66]:
# Pair each original suite with its corresponding optimized suite before pooling.
rq5_pairs = pd.concat(
    {tool: pair_runs(select_approach(tool, "mixed"),
                     select_approach(tool, "mixed", optimized=True), model_list)
     for tool in ("atgt", "evoavalla", "random")},
    names=["tool", "asm_path"],
)
report_support(rq5_pairs, "rq5_aggregate", "original", "optimized")

# Classify optimization only for suites with an original OK execution.
original_ok = rq5_pairs["status_baseline"].eq("OK")
no_valid = original_ok & rq5_pairs["status_candidate"].eq("NO_VALID_SCENARIOS")
both_ok = original_ok & rq5_pairs["status_candidate"].eq("OK")
before, after = rq5_pairs["n_scenarios_baseline"], rq5_pairs["n_scenarios_candidate"]
counts_available = before.notna() & after.notna()
fewer_tests = both_ok & counts_available & (after < before)
successful = both_ok & counts_available & (after >= before)
not_assessable = original_ok & ~(no_valid | fewer_tests | successful)
show_table(pd.DataFrame([{
    "original_OK": original_ok.sum(),
    "failed_NO_VALID_SCENARIOS": no_valid.sum(),
    "failed_fewer_tests": fewer_tests.sum(),
    "successful": successful.sum(),
    "not_assessable": not_assessable.sum(),
}]), "rq5_aggregate_optimization")

rq5_successful_pairs = rq5_pairs.loc[successful].copy()
rq5_results = compare_metrics(rq5_successful_pairs,
                              ["n_set", "n_check", "agg_cov_mean", "agg_mut_general"],
                              "rq5_aggregate")



rq5 aggregate support
   comparison  scope  original_present  optimized_present  original_OK  optimized_OK  both_OK  original_OK_retained_pct  original_OK_optimized_failed  original_OK_optimized_missing  optimized_OK_original_not_OK
rq5_aggregate    885               885                885          347           286      284                   81.8444                            63                              0                             2


rq5 aggregate transitions
   original_status   optimized_status  models
NO_VALID_SCENARIOS NO_VALID_SCENARIOS     533
                OK                 OK     284
                OK NO_VALID_SCENARIOS      63
      MUTATION_ERR       MUTATION_ERR       3
NO_VALID_SCENARIOS                 OK       2


rq5 aggregate optimization
 original_OK  failed_NO_VALID_SCENARIOS  failed_fewer_tests  successful  not_assessable
         347                         63                  31         253               0


rq5 aggregate descriptive
         metric  n